In [4]:
import numpy as np

def transpose(matrix):
    """Transpose a 3x3 matrix."""
    return np.transpose(matrix)

def LMStoDKL(lms):
    """Convert LMS values to DKL space."""
    # LMS to DKL matrix
    M_LMS_DKL = transpose(np.array([
        [1.0,  1.0,  0.0],
        [1.0, -2.31113018,  0.0],
        [-1.0, -1.0,  50.97757133]
    ]))
    # Subtract LMS gray (D65 white point) and apply transformation
    return lms @ M_LMS_DKL

def DKLtoLMS(dkl):
    """Convert DKL values to LMS space."""
    # DKL to LMS matrix (inverse of LMS to DKL)
    M_DKL_LMS = transpose(np.array([
        [0.69798832,  0.30201168,  0.0],
        [0.30201168, -0.30201168, -0.0],
        [0.01961647,  0.0,        0.01961647]
    ]))
    # Apply transformation to convert DKL to LMS
    return dkl @ M_DKL_LMS

def LMStoXYZ(lms):
    """Convert LMS values to XYZ space."""
    # LMS to XYZ transformation matrix (CIE 2006)
    M_LMS_XYZ = transpose(
        np.array([
        [2.629129278399650,  -3.780202391780134,  10.294956387893450],
        [0.865649062438827,   1.215555811642301,  -0.984175688105352],
        [-0.008886561474676,   0.081612628990755,  51.371024830897888]
    ]))
    # Perform LMS to XYZ transformation
    return lms @ M_LMS_XYZ

def xyztoRGB(xyz):
    """Convert XYZ values to RGB space."""
    # XYZ to RGB transformation matrix (sRGB/Rec.709)
    M_XYZ_RGB = transpose(np.array([
        [3.2406, -1.5372, -0.4986],
        [-0.9689,  1.8758,  0.0415],
        [0.0557, -0.2040,  1.0570]
    ]))
    # Perform XYZ to RGB transformation
    rgb = xyz @ M_XYZ_RGB
    # Clamp to the range [0, 1] for display
    return np.clip(rgb, 0.0, 1.0)

def LMStoRGB(lms):
    """Convert LMS values directly to RGB space."""
    # Step 1: Convert LMS to XYZ
    XYZ = LMStoXYZ(lms)
    print(XYZ)
    # Step 2: Convert XYZ to RGB
    return xyztoRGB(XYZ)
    
import numpy as np

# Example Usage:
LMSGray = np.array([0.739876529525622, 0.320136241543338, 0.020793708751515])  # D65 white point in LMS
DKLGray = LMStoDKL(LMSGray)
finalLMS = DKLtoLMS(DKLGray)
finalRGB = LMStoRGB(finalLMS)

print("Input LMSGray:", LMSGray)
print("After LMS → DKL:", DKLGray)
print("After DKL → LMS:", finalLMS)

print("finalRGB:", finalRGB)


[0.94912161 1.00915223 1.08774633]
Input LMSGray: [0.73987653 0.32013624 0.02079371]
After LMS → DKL: [ 1.06001277e+00 -1.69562142e-11  2.66412905e-11]
After DKL → LMS: [0.73987653 0.32013624 0.02079371]
finalRGB: [0.98210436 1.         0.99674689]


In [5]:
M_LMS_DKL = transpose(np.array([
    [1.0,  1.0,  0.0],
    [1.0, -2.31113018,  0.0],
    [-1.0, -1.0,  50.97757133]
]))

M_DKL_LMS = transpose(np.array([
    [0.69798832,  0.30201168,  0.0],
    [0.30201168, -0.30201168, -0.0],
    [0.01961647,  0.0,        0.01961647]
]))
print(np.allclose(np.linalg.inv(M_LMS_DKL), M_DKL_LMS)
)

True


In [21]:
def RGBtoXYZ(rgb):
    """Convert RGB values to XYZ space."""
    # Inverse of XYZ to RGB transformation matrix (sRGB/Rec.709)
    M_RGB_XYZ = np.linalg.inv(transpose(np.array([
        [3.2406, -1.5372, -0.4986],
        [-0.9689,  1.8758,  0.0415],
        [0.0557, -0.2040,  1.0570]
    ])))
    print(M_RGB_XYZ)
    # Perform RGB to XYZ transformation
    xyz = rgb @ M_RGB_XYZ
    return xyz

def XYZtoLMS(xyz):
    """Convert XYZ values to LMS space."""
    # Inverse of LMS to XYZ transformation matrix (CIE 2006)
    M_XYZ_LMS = np.linalg.inv(transpose(
        np.array([
        [2.629129278399650,  -3.780202391780134,  10.294956387893450],
        [0.865649062438827,   1.215555811642301,  -0.984175688105352],
        [-0.008886561474676,   0.081612628990755,  51.371024830897888]
    ])))
    # Perform XYZ to LMS transformation
    lms = xyz @ M_XYZ_LMS
    return lms

def RGBtoLMS(rgb):
    """Convert RGB values directly to LMS space."""
    # Step 1: Convert RGB to XYZ
    xyz = RGBtoXYZ(rgb)
    # Step 2: Convert XYZ to LMS
    lms = XYZtoLMS(xyz)
    return lms

print(RGBtoLMS([63/255, 146/255, 250/255]))
# print([147/255, 195/255, 255/255])
# xyz = RGBtoXYZ([147/255, 195/255, 255/255])
# print(xyz)
# rgb_reversed = xyztoRGB(xyz)
# print(rgb_reversed)

[[0.41239559 0.21258623 0.01929722]
 [0.35758343 0.7151703  0.11918386]
 [0.18049265 0.0722005  0.95049713]]
[0.37596786 0.18620636 0.01933016]
